In [2]:
import tensorflow as tf
import numpy as np
import os

# Load the trained models
efficientnet_model = tf.keras.models.load_model('true_model_version_1.keras')
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

mobilenet_model = tf.keras.models.load_model(
    'true_mobilenetv2_lichen_model_1.keras',
    custom_objects={'preprocess_input': preprocess_input}
)


# --- Re-create your test_ds if it's not directly available ---
# Assuming 'dataset' and 'partition_of_dataset' are defined as in your notebook
# and 'lichen_images' is your base directory
IMAGE_SIZE = 224
BATCH_SIZE = 32
CHANNELS = 3

full_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "lichen_images",
    shuffle=True, # Shuffle for initial loading
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE
)

def partition_of_dataset(ds, train_split=0.8, val_split=0.1, shuffle=True, shuffle_size=1000):
    ds_size = len(ds)
    if shuffle:
        ds = ds.shuffle(shuffle_size, seed=12)

    train_size = int(train_split * ds_size)
    val_size = int(val_split * ds_size)

    train_ds = ds.take(train_size)
    val_ds = ds.skip(train_size).take(val_size)
    test_ds = ds.skip(train_size).skip(val_size)

    return train_ds, val_ds, test_ds

_, _, test_ds = partition_of_dataset(full_dataset)
test_ds = test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

# Get predictions (probabilities) from each model on the test set
efficientnet_preds = efficientnet_model.predict(test_ds)
mobilenet_preds = mobilenet_model.predict(test_ds)

# Get true labels from the test dataset for evaluation
true_labels = []
for images, labels in test_ds.unbatch():
    true_labels.append(labels.numpy())
true_labels = np.array(true_labels)


Found 1824 files belonging to 10 classes.
7/7 ━━━━━━━━━━━━━━━━━━━━ 32s 1s/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 1s/step


In [3]:
# Average the probabilities
ensemble_preds = (efficientnet_preds + mobilenet_preds) / 2.0

# Get the predicted class for each image (index of the highest probability)
ensemble_predicted_classes = np.argmax(ensemble_preds, axis=1)
true_classes = true_labels # Already extracted

In [4]:
# Calculate ensemble accuracy
correct_predictions = (ensemble_predicted_classes == true_classes).sum()
total_predictions = len(true_classes)
ensemble_accuracy = correct_predictions / total_predictions

print(f"EfficientNetB0 Test Accuracy: {efficientnet_model.evaluate(test_ds)[1]*100:.2f}%")
print(f"MobileNetV2 Test Accuracy: {mobilenet_model.evaluate(test_ds)[1]*100:.2f}%")
print(f"Ensemble (Soft Voting) Test Accuracy: {ensemble_accuracy*100:.2f}%")
print(f"Ensemble (Soft Voting) Test Accuracy: {ensemble_accuracy*100:.2f}%")

7/7 ━━━━━━━━━━━━━━━━━━━━ 18s 1s/step - accuracy: 0.8209 - loss: 0.5674
EfficientNetB0 Test Accuracy: 82.14%
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.7702 - loss: 0.6987
MobileNetV2 Test Accuracy: 77.23%
Ensemble (Soft Voting) Test Accuracy: 85.71%
Ensemble (Soft Voting) Test Accuracy: 85.71%


In [1]:
# check_versions.py

try:
    import numpy as np
    print(f"numpy=={np.__version__}")
except ImportError:
    print("NumPy not found.")

try:
    import PIL
    print(f"Pillow=={PIL.__version__}")
except ImportError:
    print("Pillow not found.")

try:
    import tensorflow as tf
    print(f"tensorflow=={tf.__version__}")
except ImportError:
    print("TensorFlow not found.")

try:
    import flask
    print(f"flask=={flask.__version__}")
except ImportError:
    print("Flask not found, but that's okay, we can add a recent version.")

# Add libraries for the new web app
try:
    import fastapi
    print(f"fastapi=={fastapi.__version__}")
except ImportError:
    print("FastAPI not found, but that's okay, we can add a recent version.")

try:
    import uvicorn
    print(f"uvicorn=={uvicorn.__version__}")
except ImportError:
    print("Uvicorn not found, but that's okay, we can add a recent version.")


numpy==1.26.4
Pillow==11.2.1
tensorflow==2.19.0
flask==3.1.0
FastAPI not found, but that's okay, we can add a recent version.
Uvicorn not found, but that's okay, we can add a recent version.


C:\Users\BIT\AppData\Local\Temp\ipykernel_1508\4249549014.py:23: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.1. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print(f"flask=={flask.__version__}")
